In [112]:
import numpy as np
import pandas as pd
import  seaborn as sns
import matplotlib.pyplot as plt
from dataprep.eda.distribution.render import kde_viz


ImportError: cannot import name 'display' from 'IPython.core.display' (/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/IPython/core/display.py)

In [ ]:
df = pd.read_csv('insurance.csv')

In [ ]:
df.head()

In [ ]:
df.describe().transpose()

In [ ]:
sns.scatterplot(data=df,x=df['bmi'],y=df['charges'],hue=df['charges'])

In [ ]:
sns.displot(data=df,x=df['bmi'],kde=True)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.columns

In [ ]:
df['region'].value_counts()

In [ ]:
df = pd.get_dummies(data = df , columns=['sex','smoker','region'],drop_first=True)

In [ ]:
# The single most impactful feature on this dataset
df['bmi_smoker'] = df['bmi'] * df['smoker_yes']

# Age non-linearity — charges scale faster with age
df['age2'] = df['age'] ** 2

# Obesity flag interaction with smoker
df['obese_smoker'] = (df['bmi'] >= 30).astype(int) * df['smoker_yes']

# BMI risk tier
df['bmi_tier'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100],
                         labels=['under', 'normal', 'overweight', 'obese'])
df = pd.get_dummies(df, columns=['bmi_tier'], drop_first=True)

In [ ]:
df.head()

In [ ]:
X = df.drop('charges',axis=1)

In [ ]:
y = df['charges']

In [ ]:
X

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df.shape

In [113]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.25
)

In [114]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [115]:
models = {
    'Linear Regression': LinearRegression(),
    'Elastic Net': ElasticNet(),
    'Random Forest': RandomForestRegressor(random_state=42)
    ,'XGBoost' : XGBRegressor(random_state=42)
}


In [116]:
param_grids = {

    'Linear Regression': {},  # No hyperparameters to tune

    'Elastic Net': {
        # Wider alpha range with finer steps
        'model__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100],
        # Full spectrum coverage
        'model__l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        # Higher cap to ensure convergence on complex combos
        'model__max_iter': [5000, 10000, 20000],
        # Solvers suited for small datasets
        'model__selection': ['cyclic', 'random']
    },

    'Random Forest': {
        # More trees = more stable predictions
        'model__n_estimators': [100, 200, 300, 500, 700],
        # Deeper search including shallow trees to catch underfitting
        'model__max_depth': [None, 3, 5, 10, 15, 20, 30],
        # Finer granularity on split control
        'model__min_samples_split': [2, 3, 5, 8, 10, 15],
        # Tighter leaf control
        'model__min_samples_leaf': [1, 2, 3, 4, 6],
        # Added 'None' = all features, useful on small datasets
        'model__max_features': ['sqrt', 'log2', None],
        # Bootstrap sampling option
        'model__bootstrap': [True, False]
    },

}

param_grids['XGBoost'] = {
    # More estimators with early stopping in mind
    'model__n_estimators': [100, 200, 300, 500, 700],
    # Finer learning rate steps — critical for accuracy
    'model__learning_rate': [0.005, 0.01, 0.03, 0.05, 0.1, 0.2],
    # Wider depth range
    'model__max_depth': [2, 3, 4, 5, 6, 7, 9],
    # Finer subsample steps
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    # Finer column sampling
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    # Regularization — often ignored but very impactful
    'model__reg_alpha': [0, 0.01, 0.1, 1],       # L1
    'model__reg_lambda': [0.5, 1, 2, 5],          # L2
    # Minimum child weight — controls overfitting on small data
    'model__min_child_weight': [1, 3, 5, 7]
}
param_grids['Gradient Boosting'] = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [2, 3, 4, 5],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__max_features': ['sqrt', 'log2', None]
}


In [117]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

In [118]:
best_models = {}
results = {}

for name, model in models.items():

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    grid = RandomizedSearchCV(
        pipe,
        param_grids[name],
        cv=5,
        scoring='r2',
        n_jobs=-1
        ,random_state=42
        ,n_iter=80
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_
    results[name] = grid.best_score_

    print(f"\n{name}")
    print("Best Score:", grid.best_score_)
    print("Best Params:", grid.best_params_)
    print("-"*95)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=80. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Linear Regression
Best Score: 0.7412417234869049
Best Params: {}
-----------------------------------------------------------------------------------------------

Elastic Net
Best Score: 0.7412751004944637
Best Params: {'model__selection': 'cyclic', 'model__max_iter': 5000, 'model__l1_ratio': 0.4, 'model__alpha': 0.01}
-----------------------------------------------------------------------------------------------

Random Forest
Best Score: 0.8539486118028119
Best Params: {'model__n_estimators': 100, 'model__min_samples_split': 5, 'model__min_samples_leaf': 6, 'model__max_features': None, 'model__max_depth': 20, 'model__bootstrap': True}
-----------------------------------------------------------------------------------------------

XGBoost
Best Score: 0.8574431234851074
Best Params: {'model__subsample': 0.7, 'model__reg_lambda': 5, 'model__reg_alpha': 0.01, 'model__n_estimators': 300, 'model__min_child_weight': 3, 'model__max_depth': 2, 'model__learning_rate': 0.05, 'model__colsample_b

In [119]:
best_name = max(results, key=results.get)
final_model = best_models[best_name]

In [120]:
best_name

'XGBoost'

In [121]:
y_pred = final_model.predict(X_test)

In [122]:
from sklearn.metrics import r2_score, mean_absolute_error

print("Final Model:", best_name)
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

Final Model: XGBoost
R2: 0.8546716609168645
MAE: 2686.0830029013528
